In [1]:
from google.colab import files

uploaded = files.upload()

Saving Superstore_Analysis_With_Dashboard.xlsx to Superstore_Analysis_With_Dashboard.xlsx


In [2]:
import pandas as pd
import sqlite3

# Get uploaded file name
file_name = list(uploaded.keys())[0]

# Load only the raw Dataset sheet
df = pd.read_excel(file_name, sheet_name='Dataset')

# Create SQLite connection
conn = sqlite3.connect(':memory:')

# Save dataframe as SQL table
df.to_sql('superstore', conn, index=False, if_exists='replace')

# Preview first 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,7981,CA-2015-103800,2015-01-03,2015-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095.0,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448
1,740,CA-2015-112326,2015-01-04,2015-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784
2,741,CA-2015-112326,2015-01-04,2015-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736
3,742,CA-2015-112326,2015-01-04,2015-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540
4,1760,CA-2015-141817,2015-01-05,2015-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143.0,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536


In [3]:
# Check dataset size and column names
print("Shape:", df.shape)
print("\nColumns:")
for col in df.columns:
    print(col)

Shape: (9800, 18)

Columns:
Row ID
Order ID
Order Date
Ship Date
Ship Mode
Customer ID
Customer Name
Segment
Country
City
State
Postal Code
Region
Product ID
Category
Sub-Category
Product Name
Sales


In [4]:
query = """
SELECT
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore;
"""

pd.read_sql_query(query, conn)

,Total_Sales
0,2261536.78


In [5]:
query = """
SELECT
    COUNT(DISTINCT "Order ID") AS Total_Orders
FROM superstore;
"""

pd.read_sql_query(query, conn)

,Total_Orders
0,4922


In [6]:
query = """
SELECT
    COUNT(DISTINCT "Customer ID") AS Total_Customers
FROM superstore;
"""

pd.read_sql_query(query, conn)

,Total_Customers
0,793


In [7]:
query = """
SELECT
    Region,
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY Region
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)

,Region,Total_Sales
0,West,710219.68
1,East,669518.73
2,Central,492646.91
3,South,389151.46


In [8]:
query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY Category
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)


,Category,Total_Sales
0,Technology,827455.87
1,Furniture,728658.58
2,Office Supplies,705422.33


In [9]:
query = """
SELECT
    Segment,
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY Segment
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)


,Segment,Total_Sales
0,Consumer,1148060.53
1,Corporate,688494.07
2,Home Office,424982.18


In [10]:
query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY "Sub-Category"
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)

,Sub-Category,Total_Sales
0,Phones,327782.45
1,Chairs,322822.73
2,Storage,219343.39
3,Tables,202810.63
4,Binders,200028.79
5,Machines,189238.63
6,Accessories,164186.70
7,Copiers,146248.09
8,Bookcases,113813.20
9,Appliances,104618.40


In [11]:
query = """
SELECT
    strftime('%Y-%m', "Order Date") AS Order_Month,
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY Order_Month
ORDER BY Order_Month;
"""

pd.read_sql_query(query, conn)

,Order_Month,Total_Sales
0,2015-01,14205.71
1,2015-02,4519.89
2,2015-03,55205.80
3,2015-04,27906.85
4,2015-05,23644.30
5,2015-06,34322.94
6,2015-07,33781.54
7,2015-08,27117.54
8,2015-09,81623.53
9,2015-10,31453.39


In [12]:
query = """
SELECT
    "Customer ID",
    "Customer Name",
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY
    "Customer ID",
    "Customer Name"
ORDER BY Total_Sales DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Customer ID,Customer Name,Total_Sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.23
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.30
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


In [13]:
query = """
SELECT
    "Customer ID",
    "Customer Name",
    ROUND(SUM(Sales), 2) AS Total_Sales,
    RANK() OVER(
        ORDER BY SUM(Sales) DESC
    ) AS Customer_Rank
FROM superstore
GROUP BY
    "Customer ID",
    "Customer Name"
ORDER BY Customer_Rank
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Customer ID,Customer Name,Total_Sales,Customer_Rank
0,SM-20320,Sean Miller,25043.05,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.34,3
3,TA-21385,Tom Ashbrook,14595.62,4
4,AB-10105,Adrian Barton,14473.57,5
5,KL-16645,Ken Lonsdale,14175.23,6
6,SC-20095,Sanjit Chand,14142.33,7
7,HL-15040,Hunter Lopez,12873.30,8
8,SE-20110,Sanjit Engle,12209.44,9
9,CC-12370,Christopher Conant,12129.07,10


In [14]:
query = """
WITH customer_region_sales AS
(
    SELECT
        Region,
        "Customer ID",
        "Customer Name",
        ROUND(SUM(Sales), 2) AS Total_Sales
    FROM superstore
    GROUP BY
        Region,
        "Customer ID",
        "Customer Name"
),

ranked_customers AS
(
    SELECT
        Region,
        "Customer ID",
        "Customer Name",
        Total_Sales,
        ROW_NUMBER() OVER(
            PARTITION BY Region
            ORDER BY Total_Sales DESC
        ) AS Region_Customer_Rank
    FROM customer_region_sales
)

SELECT
    Region,
    "Customer ID",
    "Customer Name",
    Total_Sales
FROM ranked_customers
WHERE Region_Customer_Rank = 1
ORDER BY Region;
"""

pd.read_sql_query(query, conn)

,Region,Customer ID,Customer Name,Total_Sales
0,Central,TC-20980,Tamara Chand,18437.14
1,East,TA-21385,Tom Ashbrook,13723.50
2,South,SM-20320,Sean Miller,23669.20
3,West,RB-19360,Raymond Buch,14345.28


In [15]:
query = """
WITH customer_sales AS
(
    SELECT
        "Customer ID",
        "Customer Name",
        SUM(Sales) AS Total_Sales
    FROM superstore
    GROUP BY
        "Customer ID",
        "Customer Name"
)

SELECT
    "Customer ID",
    "Customer Name",
    ROUND(Total_Sales, 2) AS Total_Sales,
    ROUND(
        Total_Sales * 100.0 / SUM(Total_Sales) OVER(),
        2
    ) AS Contribution_Percentage
FROM customer_sales
ORDER BY Total_Sales DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Customer ID,Customer Name,Total_Sales,Contribution_Percentage
0,SM-20320,Sean Miller,25043.05,1.11
1,TC-20980,Tamara Chand,19052.22,0.84
2,RB-19360,Raymond Buch,15117.34,0.67
3,TA-21385,Tom Ashbrook,14595.62,0.65
4,AB-10105,Adrian Barton,14473.57,0.64
5,KL-16645,Ken Lonsdale,14175.23,0.63
6,SC-20095,Sanjit Chand,14142.33,0.63
7,HL-15040,Hunter Lopez,12873.30,0.57
8,SE-20110,Sanjit Engle,12209.44,0.54
9,CC-12370,Christopher Conant,12129.07,0.54


In [16]:
query = """
SELECT
    "Product ID",
    "Product Name",
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY
    "Product ID",
    "Product Name"
ORDER BY Total_Sales DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Product ID,Product Name,Total_Sales
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.82
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870.58
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19823.48
5,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,19024.50
6,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18839.69
7,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,18374.90
8,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,17965.07
9,OFF-SU-10000151,High Speed Automatic Electric Letter Opener,17030.31


In [17]:
query = """
SELECT
    "Ship Mode",
    ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY "Ship Mode"
ORDER BY Total_Sales DESC;
"""

pd.read_sql_query(query, conn)

,Ship Mode,Total_Sales
0,Standard Class,1340831.31
1,Second Class,449914.18
2,First Class,345572.26
3,Same Day,125219.04
